# Click prompts collector — whip videos

Runs on **olab-1's local Jupyter** (the one you already have at `http://127.0.0.1:8888/`). 
Frames stay on bigpurple. For each video we:

1. Ask bigpurple how many frames the video has (one ssh ls).
2. Rsync **just the 3 prompt frames** we need to click on (~5 MB per video).
3. You click. Left = positive, right = negative. Number keys 1/2/3 switch object id.
4. Save `prompts/<video>.json` locally and push it to bigpurple.

When you're done, on bigpurple run `sbatch --array=0-$((N-1))%8 run_inference_array.sh` to fan the inference out.

In [ ]:
%matplotlib widget
from pathlib import Path
import json, subprocess, os
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

BP_HOST      = 'bigpurple'           # ssh alias; only used when running off-cluster
BP_FRAMES    = '/gpfs/data/oermannlab/private_data/whip/frames_attempt2'
BP_REPO      = '/gpfs/data/oermannlab/users/schula12/Surgical-SAM-2'

# Detect once: if the GPFS path is reachable directly (we're on bigpurple),
# we can skip the ssh+rsync round-trip and point matplotlib straight at the
# source files. Saves time and avoids "image file is truncated" issues from
# partial rsync transfers back into the same filesystem.
ON_BIGPURPLE = Path(BP_FRAMES).is_dir()
print(f'ON_BIGPURPLE = {ON_BIGPURPLE}')

CACHE_DIR    = Path('test_data/_prompt_frames')   # only used off-cluster
CACHE_DIR.mkdir(parents=True, exist_ok=True)

PROMPTS_DIR  = Path('prompts')
PROMPTS_DIR.mkdir(exist_ok=True)

N_PROMPT_FRAMES = 3

OBJ_COLORS = ['#00ff00', '#ff8800', '#00bfff', '#ff00ff', '#ffff00',
              '#ff0000', '#00ffff', '#ffffff']

def _strip_banner(out):
    return '\n'.join(l for l in out.splitlines()
                     if 'Loading' not in l and 'requirement' not in l)

def ssh(cmd):
    """Run a command on bigpurple (off-cluster) OR locally (on-cluster)."""
    if ON_BIGPURPLE:
        r = subprocess.run(['bash', '-lc', cmd], capture_output=True, text=True)
    else:
        r = subprocess.run(['ssh', BP_HOST, cmd], capture_output=True, text=True)
    if r.returncode != 0:
        print('stderr:', r.stderr)
        r.check_returncode()
    return _strip_banner(r.stdout.strip())

def list_videos():
    """Return [(video_name, frame_count), ...] for videos with >=3 frames."""
    if ON_BIGPURPLE:
        out = []
        for d in sorted(Path(BP_FRAMES).iterdir()):
            if not d.is_dir(): continue
            if d.name.startswith('480full'): continue
            try:
                n = sum(1 for _ in d.iterdir())
            except OSError:
                continue
            if n >= 3:
                out.append((d.name, n))
        return out
    raw = ssh(f'for d in {BP_FRAMES}/*/; do '
              f'  n=$(ls $d 2>/dev/null | wc -l); '
              f'  printf "%s %d\\n" "$(basename $d)" $n; '
              f'done')
    out = []
    for line in raw.splitlines():
        parts = line.strip().rsplit(' ', 1)
        if len(parts) != 2: continue
        name, n_s = parts
        if name.startswith('480full'): continue
        try:
            n = int(n_s)
        except ValueError:
            continue
        if n >= 3:
            out.append((name, n))
    return out

videos = list_videos()
print(f'{len(videos)} videos with >=3 frames')
for v, n in videos[:5]:
    print(f'  {v}: {n} frames')
if len(videos) > 5:
    print(f'  ... and {len(videos)-5} more')

In [ ]:
def prompt_frame_indices(n_total, n_prompt=N_PROMPT_FRAMES):
    return [int(i) for i in np.linspace(0, n_total-1, n_prompt, dtype=int)]

def fetch_prompt_frames(video_name, n_total):
    """Return [(loader_idx, local_path), ...] for the 3 prompt frames.
    On bigpurple: point directly at the GPFS source (no copy).
    Off-cluster: ssh+ls to learn filenames, rsync each to a local cache."""
    idxs = prompt_frame_indices(n_total)

    if ON_BIGPURPLE:
        src_dir = Path(BP_FRAMES) / video_name
        all_files = sorted(p.name for p in src_dir.iterdir())
        assert len(all_files) == n_total, (
            f'{video_name}: expected {n_total} files, found {len(all_files)} on GPFS'
        )
        return [(i, src_dir / all_files[i]) for i in idxs]

    # Off-cluster (e.g. olab-1): ssh list, then rsync the chosen frames.
    raw = ssh(f'cd {BP_FRAMES}/{video_name} && ls | sort')
    all_files = raw.splitlines()
    assert len(all_files) == n_total, (
        f'{video_name}: expected {n_total} files, ssh ls returned {len(all_files)}'
    )
    chosen = [(i, all_files[i]) for i in idxs]
    local_dir = CACHE_DIR / video_name
    local_dir.mkdir(parents=True, exist_ok=True)
    local_paths = []
    for i, fname in chosen:
        target = local_dir / f'idx{i:07d}__{fname}'
        if not target.exists():
            subprocess.run(
                ['rsync', '-az',
                 f'{BP_HOST}:{BP_FRAMES}/{video_name}/{fname}', str(target)],
                check=True, capture_output=True)
        local_paths.append((i, target))
    return local_paths

def first_undone():
    for v, n in videos:
        if not (PROMPTS_DIR / f'{v}.json').exists():
            return v, n
    return None, None

_v, _n = first_undone()
if _v:
    print(f'Next un-done: {_v} ({_n} frames). Run the click cell below.')
else:
    print('Every video already has a prompts JSON. Nothing to do.')

In [ ]:
import io
from PIL import ImageFile as _ImageFile
import ipywidgets as widgets
from IPython.display import display
_ImageFile.LOAD_TRUNCATED_IMAGES = True

def _safe_open(image_path):
    with open(image_path, 'rb') as fp:
        data = fp.read()
    img = Image.open(io.BytesIO(data))
    img.load()
    return img


class ClickCollector:
    """Walks through a *list* of videos in sequence, single-figure, ipywidgets buttons.

    Mouse on the image:
        Left click  = positive    Right click = negative
    Buttons below:
        OBJ 1/2/3/...   pick which object the next clicks are for
        Undo            remove the most recent click
        Next/Save  ➔    advance: next prompt frame within a video,
                        next video when the last frame's done,
                        finish when the last video's last frame is done
        Skip video      give up on this video (don't save), move to next
    """

    def __init__(self, video_list, n_prompt_frames=None, push=True):
        # video_list: [(video_name, n_total), ...]
        self.video_list = list(video_list)
        self.video_idx = 0
        self.push = push
        self.n_prompt_frames = (n_prompt_frames
                                if n_prompt_frames is not None
                                else N_PROMPT_FRAMES)
        # per-video state (reset by _load_video)
        self.video_name = None; self.n_total = 0; self.chosen = []
        self.W = self.H = None
        self.step = 0; self.current_obj = 1
        self.click_history = []; self.objects_by_frame = {}
        # widget state
        self.done = False
        self.fig = None; self.ax = None
        self.status_lbl = None; self.controls = None
        self.next_btn = None; self.undo_btn = None; self.skip_btn = None
        self.obj_btns = []

    def _load_video(self, idx):
        self.video_idx = idx
        self.video_name, self.n_total = self.video_list[idx]
        self.chosen = fetch_prompt_frames(self.video_name, self.n_total)[:self.n_prompt_frames]
        first = _safe_open(self.chosen[0][1])
        self.W, self.H = first.size
        self.step = 0
        self.current_obj = 1
        self.click_history = []
        self.objects_by_frame = {}

    def _current_clicks_dict(self):
        d = {}
        for c in self.click_history:
            d.setdefault(c['oid'], {'positive': [], 'negative': []})
            d[c['oid']][c['kind']].append([c['x'], c['y']])
        return d

    def start(self):
        if not self.video_list:
            print('Nothing to do — every video already has a prompts JSON.')
            return self
        self._load_video(0)

        self.fig, self.ax = plt.subplots(figsize=(13, 7.5))
        self.fig.canvas.mpl_connect('button_press_event', self._on_click)
        self._render()
        plt.show()

        N_OBJS = 5
        self.obj_btns = []
        for i in range(1, N_OBJS+1):
            color = OBJ_COLORS[(i-1) % len(OBJ_COLORS)]
            b = widgets.Button(description=f'OBJ {i}', layout=widgets.Layout(width='80px'))
            b.style.button_color = color
            def _make_handler(oid):
                def _h(_=None):
                    if self.done: return
                    self.current_obj = oid
                    self._refresh_status()
                return _h
            b.on_click(_make_handler(i))
            self.obj_btns.append(b)

        self.undo_btn = widgets.Button(description='Undo ↶',
                                       layout=widgets.Layout(width='90px'))
        self.undo_btn.on_click(self._on_undo)
        self.skip_btn = widgets.Button(description='Skip video',
                                       layout=widgets.Layout(width='110px'))
        self.skip_btn.on_click(self._on_skip)
        self.next_btn = widgets.Button(description='Next frame / Save ➔',
                                       button_style='primary',
                                       layout=widgets.Layout(width='220px'))
        self.next_btn.on_click(self._on_next)
        self.status_lbl = widgets.HTML(value='')

        self.controls = widgets.VBox([
            widgets.HBox(self.obj_btns + [self.undo_btn, self.skip_btn, self.next_btn]),
            self.status_lbl,
        ])
        display(self.controls)
        self._refresh_status()
        return self

    def _render(self):
        loader_idx, path = self.chosen[self.step]
        img = _safe_open(path)
        self.ax.clear()
        self.ax.imshow(img)
        self.ax.axis('off')
        self.ax.set_title(
            f'{self.video_name}   ({self.video_idx+1}/{len(self.video_list)})   —  '
            f'prompt {self.step+1}/{len(self.chosen)}  '
            f'(loader idx {loader_idx})', fontsize=11)
        self.click_history = []
        self.current_obj = 1
        self.fig.canvas.draw_idle()

    def _refresh_status(self):
        if self.status_lbl is None: return
        if self.done:
            self.status_lbl.value = (
                '<div style="padding:6px;background:#666;color:white;">'
                '<b>All done</b> — every video saved. Buttons disabled.'
                '</div>'
            )
            return
        counts = {}
        for c in self.click_history:
            counts[c['oid']] = counts.get(c['oid'], 0) + 1
        counts_str = ', '.join(f'obj{oid}: {n}' for oid, n in sorted(counts.items())) or 'none'
        color = OBJ_COLORS[(self.current_obj-1) % len(OBJ_COLORS)]
        self.status_lbl.value = (
            f'<div style="padding:6px;background:{color};color:white;">'
            f'<b>OBJ {self.current_obj}</b> active &middot; '
            f'Left=positive, Right=negative &middot; '
            f'clicks so far: {counts_str}'
            f'</div>'
        )

    def _on_click(self, event):
        if self.done: return
        if event.inaxes != self.ax or event.xdata is None: return
        oid = self.current_obj
        color = OBJ_COLORS[(oid-1) % len(OBJ_COLORS)]
        if event.button == 1:
            kind = 'positive'
            marker, = self.ax.plot(event.xdata, event.ydata, marker='+',
                                   color=color, ms=22, mew=3)
            label = self.ax.annotate(f'{oid}', xy=(event.xdata, event.ydata),
                                     xytext=(8, 8), textcoords='offset points',
                                     fontsize=11, color='white',
                                     bbox=dict(facecolor=color, alpha=0.9, edgecolor='none', pad=2))
        elif event.button == 3:
            kind = 'negative'
            marker, = self.ax.plot(event.xdata, event.ydata, marker='x',
                                   color=color, ms=22, mew=3)
            label = self.ax.annotate(f'{oid}-', xy=(event.xdata, event.ydata),
                                     xytext=(8, 8), textcoords='offset points',
                                     fontsize=11, color='white',
                                     bbox=dict(facecolor=color, alpha=0.9, edgecolor='none', pad=2))
        else:
            return
        self.click_history.append({
            'oid': oid, 'kind': kind,
            'x': round(event.xdata, 1), 'y': round(event.ydata, 1),
            'marker': marker, 'label': label,
        })
        self._refresh_status()
        self.fig.canvas.draw_idle()

    def _on_undo(self, _evt=None):
        if self.done or not self.click_history: return
        last = self.click_history.pop()
        try: last['marker'].remove()
        except Exception: pass
        try: last['label'].remove()
        except Exception: pass
        self._refresh_status()
        self.fig.canvas.draw_idle()

    def _on_skip(self, _evt=None):
        if self.done: return
        print(f'Skipped {self.video_name} (no JSON saved)')
        self._advance_to_next_video()

    def _on_next(self, _evt=None):
        if self.done:
            print('(already finished)')
            return
        loader_idx = self.chosen[self.step][0]
        clicks = self._current_clicks_dict()
        if clicks:
            self.objects_by_frame[int(loader_idx)] = [
                {'obj_id': oid, 'positive': v['positive'], 'negative': v['negative']}
                for oid, v in sorted(clicks.items())
            ]
            n = sum(len(c["positive"])+len(c["negative"]) for c in self.objects_by_frame[loader_idx])
            print(f'{self.video_name} frame {loader_idx}: {n} clicks across {len(self.objects_by_frame[loader_idx])} obj(s)')
        else:
            print(f'{self.video_name} frame {loader_idx}: no clicks')
        self.step += 1
        if self.step >= len(self.chosen):
            self._save_current_video()
            self._advance_to_next_video()
        else:
            self._render()

    def _advance_to_next_video(self):
        next_idx = self.video_idx + 1
        if next_idx >= len(self.video_list):
            self._finish()
            return
        self._load_video(next_idx)
        self._render()
        self._refresh_status()

    def _save_current_video(self):
        out = {
            'video': self.video_name,
            'resolution': [self.W, self.H],
            'n_frames': self.n_total,
            'prompt_frames': [i for i, _ in self.chosen],
            'objects_by_frame': {str(k): v for k, v in self.objects_by_frame.items()},
        }
        out_path = PROMPTS_DIR / f'{self.video_name}.json'
        with open(out_path, 'w') as fp:
            json.dump(out, fp, indent=2)
        print(f'Saved {out_path}')
        if self.push and not ON_BIGPURPLE:
            subprocess.run(
                ['rsync', '-az', str(out_path), f'{BP_HOST}:{BP_REPO}/prompts/'],
                check=True)
            print(f'  pushed to {BP_HOST}:{BP_REPO}/prompts/')

    def _finish(self):
        self.done = True
        for b in self.obj_btns:
            b.disabled = True
        self.undo_btn.disabled = True
        self.skip_btn.disabled = True
        self.next_btn.disabled = True
        self._refresh_status()
        plt.close(self.fig)
        print('All videos in queue done.')


def collect_videos(video_list, n_prompt_frames=None, push=True):
    """Walk a list of (name, n_total) tuples. Use collect_undone() for "everything left"."""
    col = ClickCollector(video_list, n_prompt_frames=n_prompt_frames, push=push)
    col.start()

def collect_undone(limit=None, n_prompt_frames=None, push=True):
    """Click through every un-done video. Stop after `limit` if given."""
    todo = [(v, n) for v, n in videos if not (PROMPTS_DIR / f'{v}.json').exists()]
    if limit is not None: todo = todo[:limit]
    print(f'Will click through {len(todo)} video(s)')
    collect_videos(todo, n_prompt_frames=n_prompt_frames, push=push)

# Backwards-compatible: pass a single video and n_total.
def collect_for_video(video_name, n_total, push=True, n_prompt_frames=None):
    collect_videos([(video_name, n_total)], n_prompt_frames=n_prompt_frames, push=push)

print('Helpers loaded.  ON_BIGPURPLE =', ON_BIGPURPLE)
print('Click image, OBJ N to switch obj, Undo, Skip video, or Next/Save ➔ to advance.')

## Click through videos

Run the cell below. It pops up the first un-clicked video. Click instruments, press the OBJ buttons to switch between instruments, then click **Next frame / Save ➔**. The next un-clicked video appears automatically — no need to edit anything between videos.

- **Each saved JSON lands at**: `prompts/<video>.json` (which on bigpurple is `/gpfs/data/oermannlab/users/schula12/Surgical-SAM-2/prompts/<video>.json`). That's what `run_inference_array.sh` reads later.
- **Skip video** if you don't want to click a particular one. Nothing is saved for skipped videos.
- **Quit early** by closing the notebook tab or stopping the cell — done videos are already saved to disk; un-done ones stay un-done.

`limit=5` keeps each session short. Set to `None` to do all 38. `n_prompt_frames=1` means one prompt frame (frame 0) per video; drop the arg to get the full 3 frames at 0, N/3, 2·N/3.

In [ ]:
# Walk through every video that doesn't yet have a prompts JSON.
# Stops after `limit` videos (so you can take breaks). Set limit=None for all.
collect_undone(limit=5, n_prompt_frames=1)

# Quick progress check (run separately any time):
# done = sorted(p.stem for p in PROMPTS_DIR.glob('*.json'))
# print(f'{len(done)} / {len(videos)} done:', done)